# 05 Explanatory Model

Purpose: use panel regression and correlation analysis to understand which factors drive urban opportunity. This is an explanatory analysis, not a predictive ML model.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
scores = pd.read_csv(ROOT / 'data' / 'processed' / 'ueoi_scores.csv')
panel = pd.read_csv(ROOT / 'data' / 'raw' / 'city_panel.csv')

# Merge scores with raw panel for regression
merged = scores.merge(panel[['city', 'year', 'gdp_per_capita', 'disposable_income', 
                               'population', 'population_growth', 'innovation_index',
                               'housing_burden', 'university_resource']],
                      on=['city', 'year'], how='left')
print(f'Panel dataset: {len(merged)} rows, {len(merged["city"].unique())} cities')

## 1. UEOI Component Correlations

Which sub-scores correlate most strongly with the final UEOI score?

In [ ]:
sub_scores = ['income_score', 'gdp_score', 'population_growth_score', 
              'innovation_score', 'housing_burden_score']

corr_with_ueoi = merged[sub_scores + ['ueoi_score']].corr()['ueoi_score'].drop('ueoi_score').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['darkgreen' if v > 0 else 'darkred' for v in corr_with_ueoi.values]
bars = ax.barh(corr_with_ueoi.index, corr_with_ueoi.values, color=colors, edgecolor='white')
ax.set_xlabel('Pearson r with UEOI Score')
ax.set_title('Sub-Score Correlation with UEOI')
ax.axvline(x=0, color='gray', linewidth=0.5)
for bar, val in zip(bars, corr_with_ueoi.values):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

print('Note: high correlations are partly mechanical (UEOI is a weighted sum of these components).')
print('Income and GDP dominate due to their larger weights (0.35 and 0.25).')

## 2. Cross-Sectional Regression (by Year)

For each year, regress UEOI on raw metrics to see which real-world factors drive the index.

In [ ]:
import statsmodels.api as sm

raw_features = ['disposable_income', 'gdp_per_capita', 'population_growth', 
                'innovation_index', 'housing_burden', 'university_resource']

results = []
for year in sorted(merged['year'].unique()):
    yr_data = merged[(merged['year'] == year) & merged['ueoi_score'].notna()].dropna(subset=raw_features)
    if len(yr_data) < 12:
        continue
    
    X = yr_data[raw_features]
    X = sm.add_constant(X)
    y = yr_data['ueoi_score']
    
    model = sm.OLS(y, X).fit()
    results.append({'year': year, 'R2': model.rsquared, 'N': len(yr_data), 'model': model})
    print(f'{year}: R²={model.rsquared:.3f}, N={len(yr_data)}')
    display(pd.DataFrame({'coef': model.params, 'pval': model.pvalues}).round(4))

## 3. Panel Fixed Effects Regression

Control for unobserved city heterogeneity with city fixed effects.

In [ ]:
from linearmodels.panel import PanelOLS

try:
    panel_data = merged.set_index(['city', 'year'])
    panel_data = panel_data.dropna(subset=['ueoi_score'] + raw_features)
    
    # City fixed effects
    fe_model = PanelOLS(
        dependent=panel_data['ueoi_score'],
        exog=panel_data[raw_features],
        entity_effects=True,
        time_effects=False
    )
    fe_result = fe_model.fit()
    print(fe_result.summary)
except ImportError:
    print('linearmodels not installed. Install with: uv add linearmodels')
    print('\nFalling back to year-demeaned OLS:')
    
    # Manual demeaning: subtract city mean
    df = merged.dropna(subset=['ueoi_score'] + raw_features).copy()
    for col in raw_features + ['ueoi_score']:
        df[f'{col}_dem'] = df.groupby('city')[col].transform(lambda x: x - x.mean())
    
    X_dem = df[[f'{c}_dem' for c in raw_features]]
    y_dem = df['ueoi_score_dem']
    X_dem = sm.add_constant(X_dem)
    
    fe_proxy = sm.OLS(y_dem, X_dem).fit()
    print(fe_proxy.summary().tables[1])

## 4. Between-City Variation: What Makes Cities Different?

Regress *average* UEOI on *average* metrics across cities.

In [ ]:
city_means = merged.groupby('city')[raw_features + ['ueoi_score']].mean()

X_btw = sm.add_constant(city_means[raw_features])
y_btw = city_means['ueoi_score']

btw_model = sm.OLS(y_btw, X_btw).fit()
print(f'Between-city R² = {btw_model.rsquared:.3f}')
display(pd.DataFrame({
    'coefficient': btw_model.params,
    'p_value': btw_model.pvalues,
    'significant': btw_model.pvalues < 0.05
}).round(4))

# Scatter: predicted vs actual
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(btw_model.fittedvalues, y_btw, s=80, edgecolors='black', linewidth=0.5)
ax.plot([y_btw.min(), y_btw.max()], [y_btw.min(), y_btw.max()], 'k--', alpha=0.3)
for city in city_means.index:
    ax.annotate(city, (btw_model.fittedvalues[city], y_btw[city]), fontsize=7, ha='center',
               xytext=(0, 8), textcoords='offset points')
ax.set_xlabel('Predicted UEOI')
ax.set_ylabel('Actual UEOI')
ax.set_title(f'Between-City Fit (R²={btw_model.rsquared:.3f})')
plt.tight_layout()
plt.show()

## 5. Robustness: Rank Correlation Across Years

How stable are city rankings year-over-year?

In [ ]:
from scipy.stats import spearmanr

rank_pivot = scores.dropna(subset=['rank']).pivot(index='city', columns='year', values='rank')

years_sorted = sorted(rank_pivot.columns)
rank_corr = pd.DataFrame(index=years_sorted, columns=years_sorted, dtype=float)

for y1 in years_sorted:
    for y2 in years_sorted:
        common = rank_pivot[[y1, y2]].dropna()
        if len(common) >= 10:
            rho, _ = spearmanr(common[y1], common[y2])
            rank_corr.loc[y1, y2] = rho

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(rank_corr.astype(float), annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=0.5, vmax=1.0, linewidths=0.5, ax=ax)
ax.set_title('Spearman Rank Correlation Across Years')
plt.tight_layout()
plt.show()

print('High year-to-year rank correlation indicates stable city ordering.')
print(f'Mean Spearman ρ: {rank_corr.astype(float).mean().mean():.3f}')

## 6. Interpretation

**Key findings:**
1. **Income and GDP are the dominant drivers** of UEOI rankings (by construction, with 35% + 25% weight). Empirically, the raw correlation between disposable income and GDP per capita is >0.8—these two move together.
2. **Housing burden acts as a penalty**—cities with high absolute housing costs (Suzhou, Xiamen) can see significant UEOI deductions.
3. **Population growth and innovation have moderate impact** (15% each). These differentiate mid-tier cities.
4. **Rankings are highly stable** year-over-year (Spearman ρ > 0.90), suggesting structural rather than cyclical factors drive urban opportunity.
5. **Between-city variation explains most of the variance** (R² > 0.90), meaning the cross-sectional comparison is the primary source of information.

**Caveats:**
- This is an *index* regression, not a causal model. The sub-score correlations are partly mechanical.
- 4 cities are missing from 2025, reducing that year's N to 16.
- The Chengdu innovation index caliber issue (wide vs narrow) may affect the 2021–2023 innovation coefficient.